# GrievX — AI-Powered Public Grievance Resolution System

A step-by-step grievance resolution project in the same workflow style as the Bangalore home price notebook.

This notebook:
- loads your grievance datasets
- explores and cleans the data
- trains a Random Forest Regressor
- predicts resolution days
- routes complaints to the correct ministry and district hierarchy
- saves the trained model as a single `.joblib` file

In [ ]:
import json
from pathlib import Path
from typing import Any

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

## Load and Explore Dataset

We use three datasets:
- historical complaint data for model training
- MLA + ward member hierarchy for district routing
- minister dataset for ministry routing

In [ ]:
BASE_DIR = Path.cwd()
COMPLAINTS_FILE = BASE_DIR / 'GrievX _5 _Complaints_Dataset.xlsx'
MLA_FILE = BASE_DIR / 'Final_MLA_and_Ward_member_Dataset.csv'
MINISTER_FILE = BASE_DIR / 'tamilnadu_2021_ministers_dataset.csv'
ARTIFACT_DIR = BASE_DIR / 'artifacts'
JOBLIB_FILE = ARTIFACT_DIR / 'grievx_resolution_model.joblib'
METADATA_FILE = ARTIFACT_DIR / 'grievx_metadata.json'

complaints = pd.read_excel(COMPLAINTS_FILE)
mla = pd.read_csv(MLA_FILE)
ministers = pd.read_csv(MINISTER_FILE)

print('Complaints shape:', complaints.shape)
print('MLA/Ward shape:', mla.shape)
print('Ministers shape:', ministers.shape)

complaints.head()

In [ ]:
print('Complaint columns:', complaints.columns.tolist())
print('Missing values in complaints:\n', complaints.isna().sum())
print('\nComplaint categories:', sorted(complaints['Problem_Category'].dropna().unique().tolist()))
print('\nDistricts covered in MLA data:', sorted(mla['district'].dropna().unique().tolist()))
print('\nComplaint target summary:\n', complaints['Days_To_Complete'].describe())

## Data Cleaning and Feature Engineering

The complaint dataset already contains the target variable `Days_To_Complete`.

We will use:
- `Problem_Name` as complaint text
- `Problem_Category` as categorical context
- `Days_To_Complete` as the regression target

In [ ]:
def build_lookup_tables(complaints_df: pd.DataFrame, mla_df: pd.DataFrame, ministers_df: pd.DataFrame):
    category_lookup = {}
    for _, row in complaints_df.drop_duplicates(subset=['Problem_Category']).iterrows():
        category_lookup[str(row['Problem_Category']).strip().lower()] = {
            'Ministry': str(row['Ministry']).strip(),
            'Minister_Name': str(row['Minister_Name']).strip(),
        }

    district_lookup = {}
    for district, group in mla_df.groupby('district', dropna=False):
        district_lookup[str(district).strip().lower()] = (
            group[[
                'ac_name', 'ac_no', 'district', 'winning_cand', 'MLA_Party',
                'Panchayat_Union (Ooratchi)', 'Ward_Name', 'Ward_Member_Name',
                'Ward_Member_Party', 'Governor', 'Chief Minister'
            ]]
            .drop_duplicates()
            .to_dict(orient='records')
        )

    ministers_catalog = ministers_df[['Minister_Name', 'Ministry', 'Responsibilities']].drop_duplicates().fillna('').to_dict(orient='records')
    district_lookup['_minister_catalog'] = ministers_catalog
    return category_lookup, district_lookup

category_lookup, district_lookup = build_lookup_tables(complaints, mla, ministers)

features = complaints[['Problem_Name', 'Problem_Category']].copy()
target = complaints['Days_To_Complete'].copy()

features.head()

## Split Data into Train and Test Sets

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.2,
    random_state=42,
)

print('Train shape:', x_train.shape, y_train.shape)
print('Test shape:', x_test.shape, y_test.shape)

## Build and Train the Model

A Random Forest Regressor is used to predict the expected number of days needed to resolve a complaint.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('complaint_text', TfidfVectorizer(lowercase=True, stop_words='english', ngram_range=(1, 2)), 'Problem_Name'),
        ('category', OneHotEncoder(handle_unknown='ignore'), ['Problem_Category']),
    ],
    remainder='drop',
)

pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', RandomForestRegressor(random_state=42, n_jobs=-1)),
    ]
)

parameter_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [2, 4],
    'model__min_samples_leaf': [1, 2],
}

search = GridSearchCV(
    pipeline,
    parameter_grid,
    scoring='neg_mean_absolute_error',
    cv=5,
    n_jobs=-1,
)
search.fit(x_train, y_train)

best_model = search.best_estimator_
print('Best parameters:', search.best_params_)
print('Best cross-validation score (neg MAE):', round(search.best_score_, 4))

## Evaluate Model Performance

Because this is a regression problem, we use MAE and R2.

In [ ]:
sample_complaints = pd.DataFrame(
    [
        {'Problem_Name': 'Water leakage near street pipeline', 'Problem_Category': 'water'},
        {'Problem_Name': 'Pothole on village main road', 'Problem_Category': 'roads'},
        {'Problem_Name': 'Crop damage after rain', 'Problem_Category': 'agriculture'},
    ]
)

sample_predictions = best_model.predict(sample_complaints)
for row, predicted_days in zip(sample_complaints.to_dict(orient='records'), sample_predictions):
    print(row, '->', round(float(predicted_days), 2), 'days')

## Routing Logic

In [ ]:
def route_complaint(category_lookup: dict[str, dict[str, str]], district_lookup: dict[str, list[dict[str, Any]]], complaint_category: str, district: str) -> dict[str, Any]:
    return {
        'category': complaint_category,
        'district': district,
        'ministry_info': category_lookup.get(complaint_category.strip().lower(), {}),
        'district_hierarchy': district_lookup.get(district.strip().lower(), []),
    }

route_example = route_complaint(category_lookup, district_lookup, 'water', 'Erode')
print(json.dumps(route_example, indent=2, ensure_ascii=False))

## Save Model in Joblib Format

In [ ]:
ARTIFACT_DIR.mkdir(exist_ok=True)
joblib.dump(best_model, JOBLIB_FILE)

metadata = {
    'problem_categories': sorted(complaints['Problem_Category'].dropna().unique().tolist()),
    'problem_names': sorted(complaints['Problem_Name'].dropna().unique().tolist()),
    'districts_supported': sorted(mla['district'].dropna().unique().tolist()),
    'minister_lookup': category_lookup,
    'district_lookup': district_lookup,
    'model_file': JOBLIB_FILE.name,
}

with METADATA_FILE.open('w', encoding='utf-8') as handle:
    json.dump(metadata, handle, indent=2, ensure_ascii=False)

print('Saved model to:', JOBLIB_FILE)
print('Saved metadata to:', METADATA_FILE)